In [ ]:
"""
AI-Driven Job Matching System with Constraint Satisfaction (CSP)
Fixed Version with Sector Relationships
"""

import csv
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

class JobSeeker:
    """Represents a job seeker with robust data validation"""
    def __init__(self, skills, experience, salary, location, job_interest, sector, education_level, job_id):
        self.skills = [s.strip().lower() for s in skills if s.strip()] or ['general-skills']
        self.experience = max(0, int(experience)) if str(experience).isdigit() else 0
        self.salary = max(0, int(salary)) if str(salary).isdigit() else 0
        self.location = location.strip().lower() or 'any'
        self.job_interest = job_interest.strip().lower() or 'any'
        self.sector = sector.strip().lower() or 'general'
        self.education_level = education_level.strip().lower() or 'high school'
        self.job_id = job_id

    def __repr__(self):
        return f"JobSeeker({self.job_id})"

class JobOffer:
    """Represents a job offer with flexible constraints"""
    def __init__(self, id, required_skills, min_experience, salary_range, location, sector, education_level):
        self.id = id
        self.required_skills = self._clean_skills(required_skills)
        self.min_experience = max(0, int(min_experience)) if str(min_experience).isdigit() else 0
        self.salary_range = (
            max(0, int(salary_range[0])) if str(salary_range[0]).isdigit() else 0,
            max(int(salary_range[0]), int(salary_range[1])) if str(salary_range[1]).isdigit() else 100000
        )
        self.location = location.strip().lower() or 'any'
        self.sector = sector.strip().lower() or 'general'
        self.education_level = education_level.strip().lower() or 'high school'
        self.cluster = -1

    def _clean_skills(self, skills):
        cleaned = [s.strip().lower() for s in skills if s.strip()]
        return cleaned or ['general-skills']

    def __repr__(self):
        return f"JobOffer({self.id})"

class CSPJobMatcher:
    """Enhanced CSP-based job matching engine"""
    def __init__(self, jobs_csv_path, num_clusters=5):
        self.job_offers = self._load_job_offers(jobs_csv_path)
        self.num_clusters = num_clusters
        self.vectorizer = TfidfVectorizer(
            stop_words=None,
            token_pattern=r'(?u)\b\w+\b',
            min_df=1
        )
        self.kmeans = None
        self.job_clusters = defaultdict(list)
        self.education_levels = {
            "no formal education": 0,
            "high school": 1,
            "technical diploma": 2,
            "bachelor's": 3, "bachelor": 3,
            "ingénieur": 4,
            "master's": 5, "master": 5,
            "doctorate": 6, "phd": 6
        }
        
        # Fixed sector relationships without self-reference
        self.related_sectors = {
            'general': ['tech', 'business', 'healthcare', 'any'],
            'tech': ['information technology', 'software', 'it', 'general'],
            'business': ['finance', 'marketing', 'consulting', 'general'],
            'healthcare': ['medical', 'pharma', 'health', 'general'],
            'any': ['general', 'tech', 'business', 'healthcare']
        }
        
        self._prepare_clusters()

    def _load_job_offers(self, file_path):
        """Load job offers with enhanced validation"""
        job_offers = []
        try:
            with open(file_path, 'r') as file:
                reader = csv.DictReader(file)
                for row in reader:
                    skills = row.get('required_skills', '').split(',')
                    job_offers.append(JobOffer(
                        id=row.get('id', 'missing_job_id'),
                        required_skills=skills,
                        min_experience=row.get('min_experience', 0),
                        salary_range=(
                            row.get('salary_min', 0),
                            row.get('salary_max', 100000)
                        ),
                        location=row.get('location', 'any'),
                        sector=row.get('sector', 'general'),
                        education_level=row.get('education_level', 'high school')
                    ))
            print(f"Loaded {len(job_offers)} job offers.")
        except Exception as e:
            print(f"Error loading jobs: {e}")
        return job_offers

    def _prepare_clusters(self):
        """Cluster jobs with fallback handling"""
        try:
            skill_docs = [' '.join(job.required_skills) for job in self.job_offers]
            if not any(skill_docs):
                skill_docs = ['general-skills'] * len(self.job_offers)
            
            tfidf_matrix = self.vectorizer.fit_transform(skill_docs)
            self.kmeans = KMeans(n_clusters=self.num_clusters)
            clusters = self.kmeans.fit_predict(tfidf_matrix)
            
            for job, cluster in zip(self.job_offers, clusters):
                job.cluster = cluster
                self.job_clusters[cluster].append(job)
        except Exception as e:
            print(f"Clustering failed: {e}")
            self.job_clusters = defaultdict(list)

    def _calculate_compatibility(self, job_seeker, job_offer):
        """Enhanced scoring with fallback points"""
        try:
            score = 40 if 'general-skills' in job_offer.required_skills else 0
            
            # Experience scoring
            exp_diff = job_seeker.experience - job_offer.min_experience
            score += 20 * min(1, max(0, exp_diff) / 5)
            
            # Education scoring
            seeker_edu = self.education_levels.get(job_seeker.education_level, 1)
            job_edu = self.education_levels.get(job_offer.education_level, 1)
            score += 15 * (1 + (seeker_edu - job_edu)/6)
            
            # Sector scoring using fixed related_sectors
            if job_seeker.sector == job_offer.sector:
                score += 15
            elif job_offer.sector in self.related_sectors.get(job_seeker.sector, []):
                score += 10
            elif 'general' in [job_seeker.sector, job_offer.sector]:
                score += 5
                
            return min(100, max(20, score))
        except Exception as e:
            print(f"Scoring error: {e}")
            return 20

    def _get_valid_jobs(self, job_seeker):
        """Flexible constraint checking"""
        valid_jobs = []
        try:
            for job in self.job_offers:
                valid = True
                
                if not (job.salary_range[0] <= job_seeker.salary <= job.salary_range[1]):
                    if job.salary_range != (0, 100000):
                        valid = False
                
                seeker_edu = self.education_levels.get(job_seeker.education_level, 1)
                job_edu = self.education_levels.get(job.education_level, 1)
                if seeker_edu < job_edu and job.education_level != 'high school':
                    valid = False
                
                if valid:
                    valid_jobs.append(job)
                    
            if not valid_jobs:
                valid_jobs = [job for job in self.job_offers 
                             if 'general-skills' in job.required_skills]
                
        except Exception as e:
            print(f"Validation error: {e}")
            
        return valid_jobs

    def find_top_k_matches(self, job_seeker, k=5):
        """Guaranteed to return matches when possible"""
        valid_jobs = self._get_valid_jobs(job_seeker)
        if not valid_jobs:
            print("No valid jobs found, using fallback jobs.")
            valid_jobs = [
                JobOffer('fallback1', ['general-skills'], 0, (0, 100000), 'any', 'general', 'high school'),
                JobOffer('fallback2', ['general-skills'], 0, (0, 100000), 'any', 'general', 'high school')
            ]
            
        scored_jobs = [(job, self._calculate_compatibility(job_seeker, job)) 
                      for job in valid_jobs]
        scored_jobs.sort(key=lambda x: x[1], reverse=True)
        return scored_jobs[:k]

def load_job_seekers(filename):
    """Load seekers with enhanced error handling"""
    seekers = []
    try:
        with open(filename, 'r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                seekers.append(JobSeeker(
                    skills=row.get('skills', '').split(','),
                    experience=row.get('experience', 0),
                    salary=row.get('salary', 0),
                    location=row.get('location', 'any'),
                    job_interest=row.get('job_interest', 'any'),
                    sector=row.get('sector', 'general'),
                    education_level=row.get('education_level', 'high school'),
                    job_id=row.get('job_id', 'missing_seeker_id')
                ))
            print(f"Loaded {len(seekers)} job seekers.")
    except Exception as e:
        print(f"Error loading seekers: {e}")
    return seekers

if __name__ == "__main__":
    matcher = CSPJobMatcher("job_offers_ascii_clean.csv")
    seekers = load_job_seekers("job_seekers_with_ids.csv")
    
    if seekers:
        seeker = seekers[1]
        print(f"\nMatching for {seeker.job_id}:")
        matches = matcher.find_top_k_matches(seeker)
        
        print(f"\nTop {len(matches)} matches:")
        for idx, (job, score) in enumerate(matches, 1):
            print(f"{idx}. {job.id} ({score}%)")
            print(f"   Sector: {job.sector.capitalize()}")
            print(f"   Skills: {', '.join(job.required_skills)}")
            print(f"   Salary: ${job.salary_range[0]:,}-${job.salary_range[1]:,}")
            print(f"   Location: {job.location.capitalize()}\n")
    else:
        print("No seekers to process")

Loaded 8567 job offers.
Loaded 8124 job seekers.

Matching for missing_seeker_id:

Top 5 matches:
1. missing_job_id (70.0%)
   Sector: Construction
   Skills: general-skills
   Salary: $0-$100,000
   Location: Any

2. missing_job_id (70.0%)
   Sector: Construction
   Skills: general-skills
   Salary: $0-$100,000
   Location: Any

3. missing_job_id (70.0%)
   Sector: Construction
   Skills: general-skills
   Salary: $0-$100,000
   Location: Any

4. missing_job_id (70.0%)
   Sector: Construction
   Skills: general-skills
   Salary: $0-$100,000
   Location: Any

5. missing_job_id (70.0%)
   Sector: Construction
   Skills: general-skills
   Salary: $0-$100,000
   Location: Any



In [ ]:
pd.read_csv('jobs.csv')